# Baseline CTR Models

This notebook trains and evaluates two baseline models on synthetic ad-impression data:

1. Logistic Regression
2. XGBoost (with 3-fold CV grid search)

It reports AUC-ROC, log-loss, calibration curves, feature importance, error analysis, and saves test-set predictions for downstream model comparisons.

In [1]:
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.calibration import calibration_curve

# Ensure xgboost is available in fresh kernels.
try:
    from xgboost import XGBClassifier
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost"])
    from xgboost import XGBClassifier

RANDOM_SEED = 42

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.0 MB/s  0:00:00 eta 0:00:01



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

data_path = PROJECT_ROOT / "data" / "processed" / "ad_impressions.parquet"
if not data_path.exists():
    raise FileNotFoundError("Missing ad_impressions.parquet. Run simulator first: .venv/bin/python3 -m src.data.ad_simulator")

df = pd.read_parquet(data_path)
df = df.sort_values("timestamp").reset_index(drop=True)

required_cols = {
    "click",
    "split",
    "restaurant_city",
    "restaurant_cuisine",
    "campaign_id",
    "campaign_city",
    "campaign_cuisine",
    "time_of_day",
    "day_of_week",
    "ad_position",
    "bid_amount",
    "norm_rating",
    "price_distance",
    "is_evening",
    "is_weekend",
}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

categorical_features = [
    "restaurant_city",
    "restaurant_cuisine",
    "campaign_id",
    "campaign_city",
    "campaign_cuisine",
    "time_of_day",
    "day_of_week",
]
numeric_features = [
    "ad_position",
    "bid_amount",
    "norm_rating",
    "price_distance",
    "is_evening",
    "is_weekend",
]

feature_cols = categorical_features + numeric_features
X = df[feature_cols].copy()
y = df["click"].astype(int).copy()

train_mask = df["split"] == "train"
val_mask = df["split"] == "val"
test_mask = df["split"] == "test"

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val, y_val = X.loc[val_mask], y.loc[val_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

# Keep train/val/test protocol; combine train+val for final baseline fitting after CV on train.
X_trainval = pd.concat([X_train, X_val], axis=0)
y_trainval = pd.concat([y_train, y_val], axis=0)

print("Shapes:")
print("train:", X_train.shape, "val:", X_val.shape, "test:", X_test.shape)
print("CTR train/val/test:", y_train.mean(), y_val.mean(), y_test.mean())

Shapes:
train: (350000, 13) val: (75000, 13) test: (75000, 13)
CTR train/val/test: 0.060354285714285714 0.05964 0.059653333333333336


## 1) Feature Preprocessing

- One-hot encode sparse categorical features
- Standard-scale numerical features

In [3]:
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numeric_features),
    ]
)

preprocess

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

## 2) Logistic Regression Baseline

In [4]:
lr_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)),
    ]
)

lr_pipeline.fit(X_trainval, y_trainval)
lr_proba_test = lr_pipeline.predict_proba(X_test)[:, 1]

lr_auc = roc_auc_score(y_test, lr_proba_test)
lr_logloss = log_loss(y_test, lr_proba_test)

display(Markdown(f"**LR AUC-ROC:** `{lr_auc:.4f}`"))
display(Markdown(f"**LR Log-loss:** `{lr_logloss:.4f}`"))

**LR AUC-ROC:** `0.7742`

**LR Log-loss:** `0.1954`

In [5]:
prob_true_lr, prob_pred_lr = calibration_curve(y_test, lr_proba_test, n_bins=10, strategy="quantile")

fig_cal_lr = go.Figure()
fig_cal_lr.add_trace(go.Scatter(x=prob_pred_lr, y=prob_true_lr, mode="lines+markers", name="Logistic Regression"))
fig_cal_lr.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Perfect Calibration", line=dict(dash="dash")))
fig_cal_lr.update_layout(
    title="Calibration Curve - Logistic Regression",
    xaxis_title="Predicted probability",
    yaxis_title="Observed click rate",
)
fig_cal_lr.show()

## 3) XGBoost with 3-Fold CV (Train Split Only)

Hyperparameter grid:
- `max_depth`: [3, 5, 7]
- `n_estimators`: [100, 300]
- `learning_rate`: [0.05, 0.1]

In [6]:
xgb_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocess),
        (
            "model",
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_SEED,
                tree_method="hist",
                n_jobs=-1,
            ),
        ),
    ]
)

param_grid = {
    "model__max_depth": [3, 5, 7],
    "model__n_estimators": [100, 300],
    "model__learning_rate": [0.05, 0.1],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)

xgb_grid = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

# CV is intentionally run only on train split to avoid validation leakage.
xgb_grid.fit(X_train, y_train)

display(Markdown(f"**Best XGBoost params (CV):** `{xgb_grid.best_params_}`"))
display(Markdown(f"**Best CV AUC:** `{xgb_grid.best_score_:.4f}`"))

Fitting 3 folds for each of 12 candidates, totalling 36 fits


**Best XGBoost params (CV):** `{'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 300}`

**Best CV AUC:** `0.8052`

In [7]:
# Refit best XGBoost on train+val, then evaluate on test.
best_params = {k.replace('model__', ''): v for k, v in xgb_grid.best_params_.items()}

xgb_final = Pipeline(
    steps=[
        ("preprocess", preprocess),
        (
            "model",
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_SEED,
                tree_method="hist",
                n_jobs=-1,
                **best_params,
            ),
        ),
    ]
)

xgb_final.fit(X_trainval, y_trainval)
xgb_proba_test = xgb_final.predict_proba(X_test)[:, 1]

xgb_auc = roc_auc_score(y_test, xgb_proba_test)
xgb_logloss = log_loss(y_test, xgb_proba_test)

display(Markdown(f"**XGBoost AUC-ROC:** `{xgb_auc:.4f}`"))
display(Markdown(f"**XGBoost Log-loss:** `{xgb_logloss:.4f}`"))

**XGBoost AUC-ROC:** `0.8049`

**XGBoost Log-loss:** `0.1839`

In [8]:
prob_true_xgb, prob_pred_xgb = calibration_curve(y_test, xgb_proba_test, n_bins=10, strategy="quantile")

fig_cal_both = go.Figure()
fig_cal_both.add_trace(go.Scatter(x=prob_pred_lr, y=prob_true_lr, mode="lines+markers", name="Logistic Regression"))
fig_cal_both.add_trace(go.Scatter(x=prob_pred_xgb, y=prob_true_xgb, mode="lines+markers", name="XGBoost"))
fig_cal_both.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Perfect Calibration", line=dict(dash="dash")))
fig_cal_both.update_layout(
    title="Calibration Curves: LR vs XGBoost",
    xaxis_title="Predicted probability",
    yaxis_title="Observed click rate",
)
fig_cal_both.show()

display(Markdown(
    "Expected pattern: LR is often better calibrated out-of-the-box, while XGBoost can be sharper/miscalibrated."
))

Expected pattern: LR is often better calibrated out-of-the-box, while XGBoost can be sharper/miscalibrated.

## 4) XGBoost Feature Importance (Top 15)

In [9]:
pre = xgb_final.named_steps["preprocess"]
model = xgb_final.named_steps["model"]
feature_names = pre.get_feature_names_out()
importances = model.feature_importances_

fi = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .head(15)
    .sort_values("importance", ascending=True)
)

fig_fi = px.bar(
    fi,
    x="importance",
    y="feature",
    orientation="h",
    title="Top 15 XGBoost Features",
)
fig_fi.show()

## 5) Error Analysis

- AUC by cuisine bucket (top-5 cuisines vs long-tail)
- AUC by ad position

In [10]:
test_eval = X_test.copy()
test_eval["y_true"] = y_test.values
test_eval["lr_proba"] = lr_proba_test
test_eval["xgb_proba"] = xgb_proba_test

# Top-5 cuisines defined from train frequency.
top5_cuisines = X_train["restaurant_cuisine"].value_counts().head(5).index
test_eval["cuisine_bucket"] = np.where(
    test_eval["restaurant_cuisine"].isin(top5_cuisines),
    "top_5",
    "long_tail",
)

def safe_auc(y_true_local, y_score_local):
    if len(np.unique(y_true_local)) < 2:
        return np.nan
    return roc_auc_score(y_true_local, y_score_local)

auc_by_bucket = (
    test_eval.groupby("cuisine_bucket")
    .apply(lambda g: safe_auc(g["y_true"], g["xgb_proba"]))
    .rename("xgb_auc")
    .reset_index()
)

display(Markdown("### XGBoost AUC by cuisine bucket"))
display(auc_by_bucket)

auc_by_position = (
    test_eval.groupby("ad_position")
    .apply(lambda g: safe_auc(g["y_true"], g["xgb_proba"]))
    .rename("xgb_auc")
    .reset_index()
    .sort_values("ad_position")
)

fig_auc_pos = px.line(
    auc_by_position,
    x="ad_position",
    y="xgb_auc",
    markers=True,
    title="XGBoost AUC by Ad Position",
)
fig_auc_pos.show()

/var/folders/n2/btjhnh6s2ql_hskzbsmlrb6r0000gn/T/ipykernel_39827/1798740698.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: safe_auc(g["y_true"], g["xgb_proba"]))


### XGBoost AUC by cuisine bucket

,cuisine_bucket,xgb_auc
0,long_tail,0.794736
1,top_5,0.816519


/var/folders/n2/btjhnh6s2ql_hskzbsmlrb6r0000gn/T/ipykernel_39827/1798740698.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: safe_auc(g["y_true"], g["xgb_proba"]))


## 6) Save Baseline Test Predictions

In [11]:
pred_out = df.loc[test_mask, ["timestamp", "user_id", "business_id", "campaign_id", "ad_position", "restaurant_cuisine"]].copy()
pred_out["click"] = y_test.values
pred_out["lr_pred_proba"] = lr_proba_test
pred_out["xgb_pred_proba"] = xgb_proba_test

out_path = PROJECT_ROOT / "data" / "processed" / "baseline_test_predictions.parquet"
pred_out.to_parquet(out_path, index=False)

print(f"Saved baseline predictions: {out_path}")
pred_out.head()

Saved baseline predictions: /Users/anam/Desktop/Dev/yelp-ads-bidding-ctr/yelp-ads-bidding-ctr/data/processed/baseline_test_predictions.parquet


,timestamp,user_id,business_id,campaign_id,ad_position,restaurant_cuisine,click,lr_pred_proba,xgb_pred_proba
425000,2024-11-07 01:00:21,t5XvznprA86NtkYv_PvhKw,IHv21alrb3gqlL7S81JIfw,35,7,Chicken Wings,0,0.008384,0.009015
425001,2024-11-07 01:00:49,ehdm8q2kEU9-RQFojlcb5Q,In0D5E58MJurU_MjbfIFcA,77,7,Nightlife,0,0.006742,0.007590
425002,2024-11-07 01:01:20,WBkpkdui2ty58bnoPQKnfw,nj2Dah8x5kVwOEBZ46FX1g,57,6,Coffee & Tea,0,0.001659,0.001077
425003,2024-11-07 01:02:45,AkfmiWjr0rNbs603N1TeSA,4DLIC5fF_QXg61Hwv2BPeQ,70,4,Breakfast & Brunch,0,0.061354,0.031549
425004,2024-11-07 01:03:01,JySQipJmNa1Mv_-6k_hJ2Q,uNNIXTsRN98I0FKq74MoEw,47,6,Nightlife,0,0.018616,0.015794


In [12]:
# Acceptance summary checks
acceptance = {
    "LR_AUC_gt_0.65": bool(lr_auc > 0.65),
    "XGB_AUC_gt_0.70": bool(xgb_auc > 0.70),
}

display(Markdown("## Acceptance Checklist"))
for k, v in acceptance.items():
    display(Markdown(f"- {k}: **{v}**"))

display(Markdown(
    "Calibration expectation check is visual: LR should typically track diagonal more closely than XGBoost."
))

## Acceptance Checklist

- LR_AUC_gt_0.65: **True**

- XGB_AUC_gt_0.70: **True**

Calibration expectation check is visual: LR should typically track diagonal more closely than XGBoost.